# ctgkit — ingesting a CSV with a *datetime-string* time column

The source data (`sample_ctg.csv`) is a realistic export:

| column | content |
|---|---|
| `timestamp` | ISO-8601 datetime **strings**, e.g. `2024-06-01T09:00:00.000` |
| `fhr` | fetal heart rate, bpm (blank = signal loss) |
| `toco` | uterine activity |

It is sampled at **2 Hz** and runs for **38 minutes**.

Two facts about ctgkit drive everything below (see the README *Input format → Time
column & sampling rate* section for the full write-up):

1. **ctgkit is a fixed-rate model.** A `Signal` is *arrays + one scalar `hz`*. Every
   time-based result (durations, "variability <5 bpm for N min", decel ≥2/3/5 min,
   contractions/10 min, the 30-min contract) is computed from `sample_index / hz`. The
   per-sample timestamps are **not** used during analysis — at most they are read once
   to *infer* `hz`.
2. **`load_csv` can infer `hz`, but only from a *numeric* time column.** It cannot parse
   datetime strings, and `from_arrays` never infers at all. So a datetime column must be
   converted to seconds first, and the sampling rate derived from it.

We'll show the gotcha, then **four interchangeable ways** to handle the datetime
column, then window to 30 minutes, analyse, and plot.

In [ ]:
import io
import csv
from datetime import datetime
from pathlib import Path

import numpy as np

import ctgkit

# locate the fixture whether the notebook runs from examples/ or the repo root
CSV_PATH = next(p for p in [Path("sample_ctg.csv"), Path("examples/sample_ctg.csv")]
                if p.exists())

EXPECTED_MIN = 30.0
TOLERANCE_MIN = 2.0      # "+ what is allowed to add": only trim if total > 32 min

print("source:", CSV_PATH)
print(CSV_PATH.read_text().splitlines()[0])           # header
print(CSV_PATH.read_text().splitlines()[1])           # first data row

## 0. The gotcha — handing a datetime CSV straight to `load_csv`

Our time column is named `timestamp`, which is **not** one of ctgkit's recognised time
aliases (`time`, `time_s`, `t`, `sec`, `seconds`). So `load_csv` silently *ignores* it
and falls back to the **default 4.0 Hz** — which is wrong (the data is 2 Hz). Nothing
errors; the result is just quietly miscalibrated.

> If the column had been named `time` (a recognised alias), `load_csv` would instead try
> `float("2024-06-01T09:00:00.000")` and raise `ValueError`. Either way: **convert
> first.**

In [ ]:
naive = ctgkit.load_csv(str(CSV_PATH))
print(f"inferred hz : {naive.hz}  <-- WRONG (the 4.0 default; real rate is 2 Hz)")
print(f"thinks epoch: {naive.duration_min:.1f} min  <-- WRONG (real recording is 38 min)")
# 4560 samples / 4 Hz / 60 = 19 min, i.e. the duration is off by 2x because hz is wrong.

## Way A — Polars *(recommended)*

Parse the datetime column, convert to seconds-from-start, then infer `hz` from the
median gap between samples. We also check the spacing is regular, because ctgkit assumes
a uniform time base.

In [ ]:
import polars as pl

df = pl.read_csv(CSV_PATH, try_parse_dates=True)          # timestamp -> Datetime
df = df.with_columns(
    # seconds since the first sample (absolute origin is irrelevant: only gaps matter,
    # so epoch-seconds and seconds-from-start give the same result)
    ((pl.col("timestamp") - pl.col("timestamp").first())
        .dt.total_microseconds() / 1_000_000).alias("time_s")
)

gaps = df["time_s"].diff().drop_nulls()
period_s = float(gaps.median())                            # robust to occasional dropouts
hz_polars = 1.0 / period_s
rel_spread = float(gaps.std() / gaps.mean())               # ~0 == evenly spaced

print(f"inferred hz : {hz_polars:.3f} Hz   (period {period_s:.4f} s)")
print(f"regularity  : {rel_spread:.2%} spread "
      f"({'regular' if rel_spread < 0.05 else 'IRREGULAR -> resample first'})")

fhr = df["fhr"].to_numpy()
toco = df["toco"].to_numpy()

## Way B — Pandas

Same idea with pandas. `format="ISO8601"` parses any ISO variant (with or without
fractional seconds).

In [ ]:
import pandas as pd

pdf = pd.read_csv(CSV_PATH)
t = pd.to_datetime(pdf["timestamp"], format="ISO8601")
pdf["time_s"] = (t - t.iloc[0]).dt.total_seconds()

hz_pandas = 1.0 / pdf["time_s"].diff().dropna().median()
print(f"inferred hz : {hz_pandas:.3f} Hz")

# fhr/toco as float arrays (blank cells become NaN automatically)
fhr_pd = pdf["fhr"].to_numpy(dtype=float)
toco_pd = pdf["toco"].to_numpy(dtype=float)

## Way C — Pure standard library (no extra dependencies)

If you can't add Polars/Pandas, `csv` + `datetime.fromisoformat` is enough.

In [ ]:
with open(CSV_PATH, newline="") as f:
    records = list(csv.DictReader(f))

times = [datetime.fromisoformat(r["timestamp"]) for r in records]
t0 = times[0]
time_s = np.array([(ti - t0).total_seconds() for ti in times])

hz_stdlib = 1.0 / float(np.median(np.diff(time_s)))
print(f"inferred hz : {hz_stdlib:.3f} Hz")

fhr_std = np.array([float(r["fhr"]) if r["fhr"] else np.nan for r in records])
toco_std = np.array([float(r["toco"]) if r["toco"] else np.nan for r in records])

## Way D — Convert to a numeric-seconds CSV and let `load_csv` infer `hz`

If you'd rather not compute `hz` yourself, write the data back out with a **numeric**
`time_s` column. `load_csv` then infers the rate for you (the same median-gap idea, built
in) — no manual `hz` and no `from_arrays`.

In [ ]:
# reuse the Polars frame's numeric time_s column
numeric_csv = CSV_PATH.with_name("sample_ctg_seconds.csv")
df.select(["time_s", "fhr", "toco"]).write_csv(numeric_csv)

sig_loadcsv = ctgkit.load_csv(str(numeric_csv))
print(f"load_csv inferred hz : {sig_loadcsv.hz:.3f} Hz   (library self-calculated)")
print(f"load_csv duration    : {sig_loadcsv.duration_min:.1f} min   (correct: 38 min)")

All four ways agree on **2 Hz**. We proceed with the Polars result (`fhr`, `toco`,
`hz_polars`, and the `df` with `time_s`).

## 1. Window to the most recent 30 minutes

Purely time-based (uses `time_s`, independent of `hz`). Only trims when the recording
exceeds `EXPECTED_MIN + TOLERANCE_MIN` (32 min). Trimming **before** ctgkit means the
expensive signal processing only runs on 30 minutes.

In [ ]:
last_s = df["time_s"][-1]
total_min = last_s / 60.0

if total_min > EXPECTED_MIN + TOLERANCE_MIN:
    cutoff_s = last_s - EXPECTED_MIN * 60.0
    windowed = df.filter(pl.col("time_s") >= cutoff_s)
    print(f"trimmed {total_min:.1f} min -> most recent {EXPECTED_MIN:.0f} min")
else:
    windowed = df
    print(f"{total_min:.1f} min already within contract; no trim")

win_min = (windowed["time_s"][-1] - windowed["time_s"][0]) / 60.0
print(f"window: {win_min:.2f} min  ({windowed.height:,} samples)")

## 2. Build the Signal and analyse

Hand the windowed arrays to `from_arrays`, passing the **inferred** `hz` (never the
assumed default).

In [ ]:
signal = ctgkit.from_arrays(
    fhr=windowed["fhr"].to_numpy(),
    toco=windowed["toco"].to_numpy(),
    hz=hz_polars,                          # inferred from the timestamps
    meta={"source": str(CSV_PATH.name),
          "inferred_hz": round(hz_polars, 3),
          "windowed_from_min": round(total_min, 1)},
)

result = ctgkit.analyze(signal, guideline="figo", metadata={"oxytocin": True})
print(result.summary())

In [ ]:
# Fully JSON-serialisable, audit-ready output
import json
print(json.dumps(result.to_dict(), indent=2)[:900], "...")

## 3. Plot the trace

`ctgkit.plot()` returns a matplotlib `Figure` with the alert banner, baseline, normal
band, and flagged concern windows overlaid.

In [ ]:
%matplotlib inline
fig = ctgkit.plot(signal, result)
fig